# Answer-Bearing Chunks and Popularity Preference
Analyze top-10 answer containment using chunk-weighted popularity deciles. FEVER, HotpotQA, T-REx, and PopQA are excluded. Both random-article and random-chunk references are shown.

In [ ]:
import json
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
if repo_root.name == 'retrieval_answer_eval': repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root))
from notebooks.retrieval_answer_eval.shared_setup import *
from src.metrics.decile_utils import load_corpus_distributions
from src.process.analysis.analyse_answer_chunks import compute_non_answer_transition_matrix, load_answer_chunk_matches, summarize_non_answer_preference, summarize_right_chunks
from src.process.analysis.plot_wrong_retrieval_preference_curve import random_more_popular_baseline
K = 10
DECILE_COLUMN = 'pop_decile_chunk_weighted'
ANALYSIS_EXCLUSIONS = ['fever', 'hotpot_qa', 'trex', 'pop_qa']
questions = pd.read_parquet(RESULTS_DIR / 'cyro_qa_cache.parquet')

In [ ]:
matches_by_strategy = {}
right_chunk_summaries = {}
preference_summaries = {}
for key in ALL_STRATEGIES:
    matches = load_answer_chunk_matches(RESULTS_DIR / f'retrieved_docs_{key}.csv', questions, k=K, excluded_datasets=ANALYSIS_EXCLUSIONS, decile_column=DECILE_COLUMN)
    matches_by_strategy[key] = matches
    right_chunk_summaries[key] = summarize_right_chunks(matches)
    preference_summaries[key] = summarize_non_answer_preference(matches)
display(pd.concat([summary.assign(backend=strategy_label(key)) for key, summary in right_chunk_summaries.items()], ignore_index=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))
for key, summary in right_chunk_summaries.items():
    x = summary['target_decile'] + 1
    axes[0].plot(x, summary['mean_right_chunks'], marker='o', label=strategy_label(key), color=strategy_color(key))
    axes[1].plot(x, summary['questions_with_answer'] * 100, marker='o', label=strategy_label(key), color=strategy_color(key))
axes[0].set(title=f'Answer-Bearing Chunks in Top {K}', xlabel='Chunk-Weighted Target Decile (1=Rare, 10=Famous)', ylabel='Mean Answer-Bearing Chunks per Question', xticks=range(1, 11), ylim=(0, K))
axes[1].set(title=f'Questions with Any Answer-Bearing Chunk in Top {K}', xlabel='Chunk-Weighted Target Decile (1=Rare, 10=Famous)', ylabel='Questions (%)', xticks=range(1, 11), ylim=(0, 100))
for ax in axes: ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False); fig.tight_layout()
fig.savefig(IMAGES_DIR / f'answer_bearing_chunk_counts_top_{K}_chunk_weighted_no_popqa.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
with (COLLECTION_ROOT / 'metadata.json').open(encoding='utf-8') as metadata_file: metadata = json.load(metadata_file)
documents, chunks = load_corpus_distributions(metadata.get('corpus_stats', {}), 'chunk_weighted')
article_baseline = random_more_popular_baseline(documents / documents.sum())
chunk_baseline = random_more_popular_baseline(chunks / chunks.sum())
fig, ax = plt.subplots(figsize=(8, 5.1)); x = np.arange(1, 11)
ax.plot(x, article_baseline * 100, '--', color='#222222', linewidth=2, label='Random article baseline')
ax.plot(x, chunk_baseline * 100, ':', color='#666666', linewidth=2, label='Random chunk baseline')
for key, summary in preference_summaries.items():
    ax.errorbar(x, summary['preference'] * 100, yerr=summary['ci95'] * 100, marker='o', capsize=2, label=strategy_label(key), color=strategy_color(key))
ax.set(title=f'Non-Answer Popularity Preference, Chunk-Weighted Deciles (Top {K})', xlabel='Chunk-Weighted Target Decile (1=Rare, 10=Famous)', ylabel='Non-Answer Chunk More Popular Than Target (%)', xticks=x, ylim=(0, 100))
ax.text(0.01, 0.02, 'Excludes FEVER, HotpotQA, T-REx, and PopQA', transform=ax.transAxes, fontsize=9, color='#555555')
ax.spines[['top', 'right']].set_visible(False); ax.legend(frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5, -0.35))
fig.tight_layout(); output_path = IMAGES_DIR / f'substring_non_answer_preference_top_{K}_chunk_weighted_no_popqa.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight'); plt.show(); output_path

In [ ]:
boundaries = np.asarray(metadata['decile_boundaries_chunk_weighted'], dtype=float)
article_distribution = documents / documents.sum() * 100
chunk_distribution = chunks / chunks.sum() * 100
observed_by_strategy = {key: compute_non_answer_transition_matrix(matches_by_strategy[key], boundaries)[0] for key in ALL_STRATEGIES}
residual_limit = max(np.abs(matrix - baseline).max() for matrix in observed_by_strategy.values() for baseline in [article_distribution, chunk_distribution])
figure = plt.figure(figsize=(15, 4.3 * len(ALL_STRATEGIES) + 5), constrained_layout=True)
grid = figure.add_gridspec(len(ALL_STRATEGIES) + 1, 3)
axes = np.array([[figure.add_subplot(grid[row, column]) for column in range(3)] for row in range(len(ALL_STRATEGIES))])
for row, key in enumerate(ALL_STRATEGIES):
    observed = observed_by_strategy[key]
    panels = [(observed, 'Observed non-answer retrievals', 'Blues', 0, max(20, observed.max())), (observed - article_distribution[None, :], 'Observed minus random articles', 'RdBu_r', -residual_limit, residual_limit), (observed - chunk_distribution[None, :], 'Observed minus random chunks', 'RdBu_r', -residual_limit, residual_limit)]
    for column, (matrix, title, cmap, vmin, vmax) in enumerate(panels):
        sns.heatmap(matrix, ax=axes[row, column], cmap=cmap, vmin=vmin, vmax=vmax, center=0 if column else None, annot=True, fmt='.0f', cbar_kws={'label': '%' if column == 0 else 'Percentage-point difference'})
        axes[row, column].set_title(f'{strategy_label(key)}: {title}', fontweight='bold')
        axes[row, column].set_xlabel('Retrieved Chunk Decile (1=Rare, 10=Famous)')
        axes[row, column].set_ylabel('Target Question Decile (1=Rare, 10=Famous)')
        axes[row, column].set_xticklabels(range(1, 11)); axes[row, column].set_yticklabels(range(1, 11), rotation=0)
difference_axis = figure.add_subplot(grid[len(ALL_STRATEGIES), :])
faiss_minus_bm25 = observed_by_strategy['ivfpq_high'] - observed_by_strategy['bm25_plus']
difference_limit = max(5, np.ceil(np.abs(faiss_minus_bm25).max() / 5) * 5)
sns.heatmap(faiss_minus_bm25, ax=difference_axis, cmap='RdBu_r', center=0, vmin=-difference_limit, vmax=difference_limit, annot=True, fmt='.0f', cbar_kws={'label': 'FAISS high minus BM25+ (percentage points)'})
difference_axis.set_title('FAISS High Minus BM25+: Residual Popularity Preference After the Lexical Baseline', fontweight='bold')
difference_axis.set_xlabel('Retrieved Chunk Decile (1=Rare, 10=Famous)')
difference_axis.set_ylabel('Target Question Decile (1=Rare, 10=Famous)')
difference_axis.set_xticklabels(range(1, 11)); difference_axis.set_yticklabels(range(1, 11), rotation=0)
figure.suptitle(f'Non-Answer Popularity Transitions, Top {K} (Chunk-Weighted Deciles, No PopQA)', fontsize=15, fontweight='bold')
heatmap_path = IMAGES_DIR / f'substring_non_answer_heatmaps_top_{K}_chunk_weighted_no_popqa.png'
figure.savefig(heatmap_path, dpi=300, bbox_inches='tight'); plt.show(); heatmap_path